Example: Register to Unity Catalog

In [0]:
#%pip install "mlflow-skinny[databricks]>=2.4.1"
#dbutils.library.restartPython()

In [0]:
%pip show mlflow-skinny

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
import warnings
warnings.filterwarnings("ignore")
import mlflow
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from mlflow import MlflowClient

Define Catalog and Schema in Unity Catalog

In [0]:
catalog = "data_science"
schema = "default"

In [0]:
mlflow.set_registry_uri("databricks-uc")

Load Data

In [0]:
df = spark.sql("SELECT * FROM stroke_data")

In [0]:
from pyspark.sql.functions import when

In [0]:
df = df.withColumn("gender", when(df["gender"] == "Male", 0).when(df["gender"] == "Female", 1).otherwise(2))
df = df.withColumn("ever_married", when(df["ever_married"] == "No", 0).otherwise(1))
df = df.withColumn("work_type", when(df["work_type"] == "Private", 0)
                              .when(df["work_type"] == "Self-employed", 1)
                              .when(df["work_type"] == "Govt_job", 2)
                              .when(df["work_type"] == "children", 3)
                              .otherwise(4))
df = df.withColumn("Residence_type", when(df["Residence_type"] == "Urban", 0).otherwise(1))
df = df.withColumn("smoking_status", when(df["smoking_status"] == "Unknown", 0)
                                   .when(df["smoking_status"] == "never smoked", 1)
                                   .when(df["smoking_status"] == "formerly smoked", 2)
                                   .otherwise(3))

In [0]:
df.show(3)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import DoubleType

# Assuming 'bmi' column is of string type and contains numeric values as strings
df = df.withColumn("bmi", df["bmi"].cast(DoubleType()))

# Show the DataFrame after transformation
df.show(3)

In [0]:
# Drop rows with missing values
df = df.dropna()

# Show the DataFrame after dropping rows with missing values
df.show(3)

In [0]:
# Select features and target
featureCols = [col for col in df.columns if col != "stroke"]
assembler = VectorAssembler(inputCols=featureCols, outputCol="features")
data_prepared = assembler.transform(df).select(col("features"), col("stroke").alias("label"))

# Split the data
(train_data, test_data) = data_prepared.randomSplit([0.7, 0.3])
     

Train model

In [0]:
# Train a RandomForest model
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)
model = rf.fit(train_data)
model_name = 'random_forest_classifier'
# Make predictions
predictions = model.transform(test_data)
# Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy = {accuracy}")


Model signature

In [0]:
# Sample Data
sample = train_data.limit(5)
# Convert Spark DataFrame to Pandas DataFrame
train_data_pd = sample.toPandas()
# Infer model signature using the Pandas DataFrame
signature = infer_signature(train_data_pd, model.transform(sample).toPandas())

In [0]:
train_data_pd.head()

Log with MlFlow

In [0]:
with mlflow.start_run():
    mlflow.spark.log_model(model, model_name, signature=signature)
    uri = mlflow.get_artifact_uri(model_name)
    # Log metrics
    mlflow.log_metric("accuracy_score", accuracy)
    mlflow.set_registry_uri("databricks-uc")
    # Register Model
    mlflow.register_model(
        model_uri=uri,
        name=f"{catalog}.{schema}.{model_name}",
    )
    

Create Alias for latest version of model

In [0]:
# Initialize the MLflow client
client = MlflowClient()

# Search for all versions of the model and fetch the latest one
model_version_infos = client.search_model_versions(f"name='{catalog}.{schema}.{model_name}'")
new_model_version = max(model_version_info.version for model_version_info in model_version_infos)

# Set the alias for the latest model version
client.set_registered_model_alias(
    name=f"{catalog}.{schema}.{model_name}",
    alias="Best",
    version=new_model_version
)
     

Load Model

In [0]:
model_version_uri = 'models:/'+f"{catalog}.{schema}.{model_name}@Best"
best_version = mlflow.spark.load_model(model_version_uri)

In [0]:
display(best_version.transform(test_data).limit(5))